# Support Vector Classifiers

Here we try many support vector classifiers, with different kernels. 

In [ ]:
import pandas as pd

learn_data = pd.read_csv("../data/scaled_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,0.173844,1.109406,0.362037,0.487504,-0.903219,-1.399329,1.203258,0,0
1,-0.379308,0.226878,-0.573270,0.288250,1.482342,1.488320,0.927290,0,0
2,-1.362690,-0.430090,-0.232378,0.575302,-0.024328,0.212382,-0.353374,0,0
3,-0.194924,-0.795164,-0.925510,0.589292,0.101228,0.413846,-0.458710,1,0
4,0.542612,2.761282,1.783803,-0.293097,0.352339,-0.459164,1.153956,1,0


In [33]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
Xnum = X.drop(columns = ["Female"])
y = learn_data["Target"]

X_train, X_val, Xnum_train, Xnum_val, y_train, y_val = train_test_split(X, Xnum, y, test_size = 0.33, random_state = 42)

## Metrics

In [34]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear kernel (or no kernel)

We have seen with other linear classifiers that the performance is not very good because of two reasons:
- Excessive resampling: we might be resampling too much, and this may affect our predictive power by creating samples that do not correspond to the real data.

In [50]:
from sklearn.svm import LinearSVC

linear_model = LinearSVC(class_weight = "balanced")
linear_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(linear_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	73	11
	0	87	129
Accuracy: 67.33%


In [51]:
confusion(np.array(y_val), pd.Series(linear_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	32	10
	0	38	69
Accuracy: 67.79%


In [54]:
from sklearn.model_selection import cross_validate

cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)

for C in Cs:
    linear_model = LinearSVC(C = C, class_weight = "balanced")
    this_results = pd.DataFrame(cross_validate(linear_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[C, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
100.000000,0.656571,0.722822,0.680818,0.670487
86.851137,0.656571,0.722822,0.680818,0.670487
75.431201,0.656571,0.722822,0.680818,0.670487
65.512856,0.656571,0.722822,0.680818,0.670487
56.898660,0.656571,0.722822,0.680818,0.670487


In [56]:
linear_model_best = LinearSVC(C = 100, class_weight = "balanced")
cross_val_results = pd.DataFrame(cross_validate(linear_model_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Linear-best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear-best,0.656571,0.722822,0.680818,0.670487
Gaussian-Scale-Best,0.572013,0.575279,0.570632,0.654757
Gaussian-Scale,0.418042,0.488471,0.382183,0.699376
Gaussian-Auto,0.413755,0.490673,0.357713,0.706042


## Gaussian kernel

In [57]:
from sklearn.svm import SVC

rbf_scale_model = SVC(kernel = "rbf", gamma = "scale", class_weight = "balanced")
rbf_scale_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(rbf_scale_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	79	5
	0	84	132
Accuracy: 70.33%


In [58]:
confusion(np.array(y_val), pd.Series(rbf_scale_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	33	9
	0	39	68
Accuracy: 67.79%


In [59]:
rbf_scale_model = SVC(kernel = "rbf", gamma = "scale", class_weight = "balanced")
cross_val_results = pd.DataFrame(cross_validate(rbf_scale_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Gaussian-Scale", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear-best,0.656571,0.722822,0.680818,0.670487
Gaussian-Scale,0.631849,0.707245,0.669988,0.641498
Gaussian-Scale-Best,0.572013,0.575279,0.570632,0.654757
Gaussian-Auto,0.413755,0.490673,0.357713,0.706042


We can also use the automatic $\gamma$, which is $\gamma = \frac{1}{n}$. When doing simple train-validation, this gives us slightly worse results, but it isn't very 

In [60]:
rbf_auto_model = SVC(kernel = "rbf", gamma = "auto", class_weight = "balanced")
rbf_auto_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(rbf_auto_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	79	5
	0	84	132
Accuracy: 70.33%


In [61]:
confusion(np.array(y_val), pd.Series(rbf_auto_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	33	9
	0	39	68
Accuracy: 67.79%


In [62]:
rbf_auto_model = SVC(kernel = "rbf", gamma = "auto", class_weight = "balanced")
cross_val_results = pd.DataFrame(cross_validate(rbf_auto_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Gaussian-Auto", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear-best,0.656571,0.722822,0.680818,0.670487
Gaussian-Auto,0.632339,0.709707,0.672476,0.641498
Gaussian-Scale,0.631849,0.707245,0.669988,0.641498
Gaussian-Scale-Best,0.572013,0.575279,0.570632,0.654757


We can also try to find the best value of $C$, using the scaled value of $\gamma$.

In [63]:
import warnings
warnings.filterwarnings("ignore")

cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)

for C in Cs:
    rbf_model = SVC(kernel = "rbf", C = C, gamma = "scale", class_weight = "balanced")
    this_results = pd.DataFrame(cross_validate(rbf_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[C, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
3.906940,0.65788,0.721365,0.679745,0.672659
3.393222,0.65254,0.719187,0.67804,0.665968
6.866488,0.65135,0.705106,0.666953,0.670437
4.498433,0.649762,0.709519,0.67,0.665993
5.179475,0.647252,0.705519,0.666578,0.66377


In [64]:
rbf_model_best = SVC(kernel = "rbf", C = 3.9, gamma = "scale", class_weight = "balanced")
cross_val_results = pd.DataFrame(cross_validate(rbf_model_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Gaussian-Scale-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian-Scale-Best,0.65788,0.721365,0.679745,0.672659
Linear-best,0.656571,0.722822,0.680818,0.670487
Gaussian-Auto,0.632339,0.709707,0.672476,0.641498
Gaussian-Scale,0.631849,0.707245,0.669988,0.641498


## Polynomial kernel

In [65]:
poly_model = SVC(kernel = "poly", degree = 2, gamma = "scale", class_weight = "balanced")
poly_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(poly_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	73	11
	0	128	88
Accuracy: 53.67%


In [66]:
confusion(np.array(y_val), pd.Series(poly_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	34	8
	0	63	44
Accuracy: 52.35%


In [67]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)
degrees = [2, 3, 4, 5]

for degree in degrees:
    for C in Cs:
        poly_model = SVC(kernel = "poly", C = C, degree = degree, gamma = "scale", class_weight = "balanced")
        this_results = pd.DataFrame(cross_validate(poly_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
        mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
        cross_val_results.loc[f"Degree: {degree} - C: {C} ", :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
Degree: 3 - C: 2.94705170255181,0.632531,0.703933,0.667486,0.643795
Degree: 3 - C: 3.906939937054617,0.62971,0.699957,0.664877,0.641573
Degree: 3 - C: 3.3932217718953277,0.628569,0.700856,0.665219,0.639351
Degree: 3 - C: 2.5595479226995357,0.628442,0.700808,0.665005,0.639301
Degree: 3 - C: 9.102981779915218,0.626697,0.682135,0.647816,0.64377


In [68]:
poly_model_best = SVC(kernel = "poly", C = 2.9, degree = 3, gamma = "scale", class_weight = "balanced")
cross_val_results = pd.DataFrame(cross_validate(poly_model_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Poly-Degree4", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian-Scale-Best,0.65788,0.721365,0.679745,0.672659
Linear-best,0.656571,0.722822,0.680818,0.670487
Poly-Degree4,0.632531,0.703933,0.667486,0.643795
Gaussian-Auto,0.632339,0.709707,0.672476,0.641498
Gaussian-Scale,0.631849,0.707245,0.669988,0.641498


In [86]:
poly_model_best.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(poly_model_best.predict(Xnum_train)))
confusion(np.array(y_val), pd.Series(poly_model_best.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	73	11
	0	63	153
Accuracy: 75.33%
		Predicted
		+1	0
Real	+1	29	13
	0	30	77
Accuracy: 71.14%


## Sigmoid

In [69]:
sig_model = SVC(kernel = "sigmoid", gamma = "scale", class_weight = "balanced")
sig_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(sig_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	54	30
	0	85	131
Accuracy: 61.67%


In [70]:
confusion(np.array(y_val), pd.Series(sig_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	26	16
	0	37	70
Accuracy: 64.43%


In [71]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)

for C in Cs:
    sig_model = SVC(kernel = "sigmoid", C = C, gamma = "scale", class_weight = "balanced")
    this_results = pd.DataFrame(cross_validate(sig_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[C, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
0.471487,0.628644,0.700774,0.664927,0.639326
0.542868,0.623568,0.692702,0.657626,0.634856
0.355648,0.622465,0.701909,0.666889,0.630387
0.409492,0.621223,0.69701,0.662458,0.630362
4.498433,0.620681,0.655582,0.628539,0.650462


In [94]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

coeff0s = np.logspace(start = -1, stop = 1, num = 50)

for i in [-1, +1]:
    for coeff0 in coeff0s:
        sig_model = SVC(kernel = "sigmoid", C = 0.5, coef0 = i * coeff0, gamma = "scale", class_weight = "balanced")
        this_results = pd.DataFrame(cross_validate(sig_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
        mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
        cross_val_results.loc[i * coeff0, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
0.308884,0.643629,0.693486,0.657225,0.663795
2.023590,0.634974,0.654375,0.635481,0.677104
0.159986,0.633796,0.691591,0.656146,0.650437
0.255955,0.633258,0.689024,0.65392,0.650462
0.175751,0.631899,0.690077,0.654851,0.648215


In [95]:
sig_model_best = SVC(kernel = "sigmoid", C = 0.5, coef0 = 0.3, gamma = "scale", class_weight = "balanced")
cross_val_results = pd.DataFrame(cross_validate(sig_model_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Sigmoid-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian-Scale-Best,0.65788,0.721365,0.679745,0.672659
Linear-best,0.656571,0.722822,0.680818,0.670487
Sigmoid-Best,0.641532,0.691947,0.656025,0.661573
Poly-Degree4,0.632531,0.703933,0.667486,0.643795
Gaussian-Auto,0.632339,0.709707,0.672476,0.641498
Gaussian-Scale,0.631849,0.707245,0.669988,0.641498


## Trying our best models on test dataset


In [ ]:
test_data = pd.read_csv("../data/scaled_test_fs.csv", header = None)
test_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female
0,-2.100226,-0.795164,1.907027,-0.567456,1.356786,1.555475,-1.512071,0
1,1.034303,0.171538,-0.117670,1.320148,1.105674,-0.459164,1.121330,0
2,0.911380,-0.795164,-0.643898,-1.387576,1.356786,0.548155,-0.458710,0
3,0.911380,1.351361,-0.212816,3.236675,0.101228,-0.526319,1.056650,1
4,0.173844,-0.537931,-0.631959,0.132669,-0.526551,-0.123391,-0.926871,1


In [74]:
test_data_num = test_data.drop(columns = ["Female"])

In [ ]:
test_y = pd.read_csv("../data/test_y.csv").iloc[:, 1]
test_y

0      1
1      0
2      1
3      0
4      1
      ..
111    1
112    0
113    1
114    0
115    0
Name: Label, Length: 116, dtype: int64

### Gaussian kernel (no categorical)

In [80]:
rbf_model_best = SVC(kernel = "rbf", C = 100, gamma = "scale", class_weight = "balanced")
rbf_model_best.fit(Xnum, y)

labels_rbf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rbf['Label'] = pd.DataFrame(rbf_model_best.predict(test_data_num))
labels_rbf['ID'] = labels_rbf.index + 1
labels_rbf

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,0
112,113,0
113,114,1
114,115,0


In [83]:
confusion(test_y, labels_rbf["Label"])

		Predicted
		+1	0
Real	+1	15	18
	0	20	63
Accuracy: 67.24%


In [85]:
compute_metrics(test_y, labels_rbf['Label'])

[0.6047345767575323,
 0.6067907995618839,
 0.6031746031746031,
 0.6724137931034483]

In [ ]:
labels_rbf.to_csv('../data/new_predictions/svm_rbf_best.csv', index = False)

### Polynomial kernel

In [89]:
poly_model_best = SVC(kernel = "poly", C = 2.9, degree = 3, gamma = "scale", class_weight = "balanced")
poly_model_best.fit(Xnum, y)

labels_poly = pd.DataFrame(columns = ['ID', 'Label'])
labels_poly['Label'] = pd.DataFrame(poly_model_best.predict(test_data_num))
labels_poly['ID'] = labels_rbf.index + 1
labels_poly

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


In [90]:
confusion(test_y, labels_poly["Label"])

		Predicted
		+1	0
Real	+1	28	5
	0	32	51
Accuracy: 68.10%


In [91]:
compute_metrics(test_y, labels_poly['Label'])

[0.6679817436373482,
 0.7314713399050748,
 0.6886904761904762,
 0.6810344827586207]

### Sigmoid kernel

In [97]:
sig_model_best = SVC(kernel = "sigmoid", C = 0.5, coef0 = 0.3, gamma = "scale", class_weight = "balanced")
sig_model_best.fit(Xnum, y)

labels_sig = pd.DataFrame(columns = ['ID', 'Label'])
labels_sig['Label'] = pd.DataFrame(sig_model_best.predict(test_data_num))
labels_sig['ID'] = labels_sig.index + 1
labels_sig

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,0
112,113,0
113,114,1
114,115,0


In [98]:
confusion(test_y, labels_sig["Label"])

		Predicted
		+1	0
Real	+1	22	11
	0	29	54
Accuracy: 65.52%


In [99]:
compute_metrics(test_y, labels_sig['Label'])

[0.6267696267696268,
 0.6586345381526104,
 0.6310708898944193,
 0.6551724137931034]